# Lab 3 — ชื่อ model, provider, quantization และ modality (EP01)

ใน Lab 2 เราล็อก provider ไว้เจ้าเดียว เพราะ Gemma ตัวเดียวกันแต่ต่างเจ้าให้ความน่าจะเป็นไม่เท่ากัน lab นี้เราจะดูว่าทำไม

**สิ่งที่จะได้เรียน**
1. อ่านชื่อ model ด้วยโค้ด และคำนวณหน่วยความจำที่แต่ละความละเอียดต้องใช้
2. เปิดรายชื่อ provider ของ Gemma: quantization ราคา และ context window
3. ถาม Gemma คำถามเดียวกันจากหลายเจ้า แล้วเทียบความน่าจะเป็น
4. เช็ก modality ก่อนส่งรูป — Gemma รับได้ Llama 3.1 ไม่ได้

ข้อมูล provider และ modality มาจาก OpenRouter (ตำราไม่ได้ลงรายละเอียดนี้) — ตัวเลขเปลี่ยนได้ตามวันที่รัน · กราฟในส่วนที่ 5 เป็น **ข้อมูลจำลองเพื่อการสอน**

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.8, §1.10, §1.14 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **4 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep01-lab3-model-names.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่อง

ต้องมี Python 3.10+ (หรือ Colab) และ `OPENROUTER_API_KEY` — วิธีตั้งค่า key และเหตุผลที่เลือก model นี้อยู่ใน **Lab 1**

In [ ]:
%pip install -q "openai>=1.50" python-dotenv requests matplotlib

In [ ]:
# ---- จาก Lab 1: ตั้งค่า client ----
import os, io, math, base64, requests
from collections import Counter
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
LLAMA = "meta-llama/llama-3.1-8b-instruct"
EXTRA = {"reasoning": {"enabled": False}}

## 1. อ่านชื่อ model

`<ผู้สร้าง>/<ตระกูล>-<รุ่น>-<ขนาด>-…-<it/instruct>` · `26b` = 26,000 ล้าน parameter · `a4b` = MoE ทำงานจริง ~4B ต่อ token · `it` = instruction-tuned

In [ ]:
maker, name = MODEL.split("/")
print(maker, name.split("-"))

In [ ]:
def params_b(model_id):
    for part in model_id.split("/")[1].split("-"):
        if part.endswith("b") and part[:-1].replace(".", "").isdigit():
            return float(part[:-1])

for mid in [MODEL, LLAMA]:
    print(f"{mid}: {params_b(mid)} พันล้าน parameter")

## 2. หน่วยความจำกับ quantization

หน่วยความจำ (GB) ≈ จำนวน parameter (พันล้าน) × จำนวนบิต ÷ 8 — นับเฉพาะตัว model ยังไม่รวมพื้นที่ทำงานระหว่างตอบ

In [ ]:
# ✏️ TODO 1 (▶ วิดีโอ 2:40): หน่วยความจำ (GB) = parameter (พันล้าน) × จำนวนบิต ÷ 8
def memory_gb(params_b, bits):
    return ___

for q, bits in [("bf16", 16), ("fp8", 8), ("fp4", 4)]:
    print(f"{q}: {memory_gb(params_b(MODEL), bits):.0f} GB")

## 3. provider ของ Gemma

`https://openrouter.ai/api/v1/models/<model>/endpoints` บอกทุกเจ้าที่ให้บริการ model นี้ พร้อม quantization ราคา และ context window

In [ ]:
url = f"https://openrouter.ai/api/v1/models/{MODEL}/endpoints"
endpoints = requests.get(url).json()["data"]["endpoints"]
for e in sorted(endpoints, key=lambda e: float(e["pricing"]["prompt"])):
    price = float(e["pricing"]["prompt"]) * 1e6
    print(f"{e['provider_name']:<14} {e['quantization']:<8} ${price:.3f}/M  context {e['context_length']:,}")

In [ ]:
# ✏️ TODO 2 (▶ วิดีโอ 4:33): นับจำนวน provider ของแต่ละ quantization ด้วย Counter
print(len(endpoints), "provider:", Counter(___ for e in endpoints))

## 4. ชื่อเดียวกัน คำตอบไม่เท่ากัน

ถาม "I enjoy eating" แบบ Lab 2 แต่ล็อกทีละเจ้า แล้วเทียบความน่าจะเป็นของ `pizza` กับ `food`

In [ ]:
def first_token_probs(provider):
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "Complete with one word only. I enjoy eating"}],
        max_tokens=1, logprobs=True, top_logprobs=5,
        # ✏️ TODO 3 (▶ วิดีโอ 5:12): ล็อกให้ใช้ provider ที่ส่งเข้ามาเจ้าเดียว และห้ามสลับไปเจ้าอื่น
        extra_body={**EXTRA, "provider": ___},
    )
    top = resp.choices[0].logprobs.content[0].top_logprobs
    return {t.token: math.exp(t.logprob) for t in top}

In [ ]:
for p in ["google-vertex/global", "nextbit/bf16", "akashml/fp4", "darkbloom"]:
    probs = first_token_probs(p)
    print(f"{p:<22} pizza {probs.get('pizza', 0):.2f} · food {probs.get('food', 0):.2f}")

In [ ]:
r = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Hi"}], max_tokens=5,
    extra_body={**EXTRA, "provider": {"quantizations": ["bf16"]}},
)
print("served by:", r.provider)

## 5. Modality: model ไหนรับรูปได้

รายชื่อ model ของ OpenRouter มี `architecture.input_modalities` (รับอะไรได้) และ `output_modalities` (ตอบเป็นอะไร)

In [ ]:
models = requests.get("https://openrouter.ai/api/v1/models").json()["data"]
text_only = [m["id"] for m in models if m["architecture"]["input_modalities"] == ["text"]]
print(len(models), "models ·", len(text_only), "รับแค่ข้อความ")

In [ ]:
# ✏️ TODO 4 (▶ วิดีโอ 7:36): คืน True ถ้า kind อยู่ใน input_modalities ของ model
def accepts(model_id, kind):
    m = next(x for x in models if x["id"] == model_id)
    return ___

for mid in [MODEL, LLAMA]:
    m = next(x for x in models if x["id"] == mid)
    print(mid, "· รูป:", accepts(mid, "image"), "· ตอบเป็น:", m["architecture"]["output_modalities"])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 2.4))
ax.bar(["2021", "2022", "2023", "2024"], [12, 30, 85, 160])
ax.set_title("Papers on LLM agents")
buf = io.BytesIO()
fig.savefig(buf, format="png")
IMAGE = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

In [ ]:
question = [
    {"type": "text", "text": "ปีไหนมีจำนวน paper มากที่สุด ตอบสั้นๆ"},
    {"type": "image_url", "image_url": {"url": IMAGE}},
]
for mid in [MODEL, LLAMA]:
    try:
        r = client.chat.completions.create(model=mid, messages=[{"role": "user", "content": question}],
                                           max_tokens=40, extra_body=EXTRA)
        print(mid, "→", r.choices[0].message.content, "·", r.usage.prompt_tokens, "token เข้า")
    except Exception as e:
        print(mid, "→ error:", str(e)[:90])

## สรุป

| เราได้เห็น | จาก lab นี้ |
|---|---|
| ชื่อ model บอกผู้สร้าง ขนาด และรุ่น | `google` · `26b` · `a4b` · `it` |
| quantization ลดหน่วยความจำ | 52 → 26 → 13 GB |
| provider ชื่อเดียวกัน แต่ราคา ความละเอียด context ต่างกัน | 15 เจ้า · ราคาต่างเกือบ 4 เท่า · context 131K หรือ 262K |
| ต่างเจ้า คำตอบต่างกัน | Google: pizza ≈ 0.76 · Darkbloom: food ≈ 0.8 |
| เช็ก modality ก่อนส่งรูป | Gemma ตอบ 2024 (~280 token) · Llama error 404 |

**แบบฝึกหัด:** หา model บน OpenRouter ที่รับ `audio` ได้ (ใช้ `accepts`) แล้วอ่านชื่อมันด้วย `split` — ชื่อบอกขนาดไหม?

▶ **ถัดไป: สรุป EP01** ทบทวนทุกอย่างที่เราเห็นในสาม lab